

```
# This is formatted as code
```

# Lab 5
: Seq2Seq and Visual Attention




## Cell 1: Import Libraries


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    AdditiveAttention,
    Concatenate,
    Dense,
    Embedding,
    Input,
    Lambda,
    LSTM,
    Reshape,
    Softmax,
)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

np.random.seed(42)
tf.random.set_seed(42)


## Cell 2: Prepare Seq2Seq Translation Dataset


In [ ]:
input_texts = [
    "i am happy",
    "i am sad",
    "he is good",
    "she is smart",
    "we love ai",
    "they play football",
]

target_texts = [
    "je suis heureux",
    "je suis triste",
    "il est bon",
    "elle est intelligente",
    "nous aimons ia",
    "ils jouent football",
]

target_texts_in = ["<start> " + text for text in target_texts]
target_texts_out = [text + " <end>" for text in target_texts]


## Cell 3: Tokenize and Pad Seq2Seq Data


In [ ]:
input_tokenizer = Tokenizer(filters="")
target_tokenizer = Tokenizer(filters="")

input_tokenizer.fit_on_texts(input_texts)
target_tokenizer.fit_on_texts(target_texts_in + target_texts_out)

encoder_sequences = input_tokenizer.texts_to_sequences(input_texts)
decoder_input_sequences = target_tokenizer.texts_to_sequences(target_texts_in)
decoder_output_sequences = target_tokenizer.texts_to_sequences(target_texts_out)

encoder_max_len = max(len(seq) for seq in encoder_sequences)
decoder_max_len = max(len(seq) for seq in decoder_input_sequences)

encoder_input_data = pad_sequences(encoder_sequences, maxlen=encoder_max_len, padding="post")
decoder_input_data = pad_sequences(decoder_input_sequences, maxlen=decoder_max_len, padding="post")
decoder_output_data = pad_sequences(decoder_output_sequences, maxlen=decoder_max_len, padding="post")

input_vocab_size = len(input_tokenizer.word_index) + 1
target_vocab_size = len(target_tokenizer.word_index) + 1

print("Input vocab size:", input_vocab_size)
print("Target vocab size:", target_vocab_size)


Input vocab size: 16
Target vocab size: 18


## Cell 4: Build Encoder-Decoder Model With Attention


In [ ]:
embedding_dim = 64
latent_dim = 128

encoder_inputs = Input(shape=(encoder_max_len,), name="encoder_inputs")
encoder_embedding = Embedding(input_vocab_size, embedding_dim, mask_zero=True)(encoder_inputs)
encoder_outputs, state_h, state_c = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="encoder_lstm",
)(encoder_embedding)

decoder_inputs = Input(shape=(decoder_max_len,), name="decoder_inputs")
decoder_embedding = Embedding(target_vocab_size, embedding_dim, mask_zero=True)(decoder_inputs)
decoder_outputs, _, _ = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm",
)(decoder_embedding, initial_state=[state_h, state_c])

attention_outputs = AdditiveAttention(name="attention")([decoder_outputs, encoder_outputs])
decoder_context = Concatenate(axis=-1)([decoder_outputs, attention_outputs])
decoder_predictions = Dense(target_vocab_size, activation="softmax", name="output_dense")(decoder_context)

seq2seq_model = Model([encoder_inputs, decoder_inputs], decoder_predictions)
seq2seq_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
seq2seq_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 3, 64)     │      1,024 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 3)         │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 4, 64)     │      1,152 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 3, 128),  │     98,816 │ embedding[0][0],  │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 4, 128),  │     98,816 │ embedding_1[0][0… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 4, 128)    │        128 │ decoder_lstm[0][… │
│ (AdditiveAttention) │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 4, 256)    │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_dense        │ (None, 4, 18)     │      4,626 │ concatenate[0][0] │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 204,562 (799.07 KB)

 Trainable params: 204,562 (799.07 KB)

 Non-trainable params: 0 (0.00 B)

## Cell 5: Train Seq2Seq Model


In [ ]:
seq2seq_model.fit(
    [encoder_input_data, decoder_input_data],
    np.expand_dims(decoder_output_data, -1),
    batch_size=2,
    epochs=300,
    verbose=1,
)


Epoch 1/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.1250 - loss: 2.8896   
Epoch 2/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2500 - loss: 2.8727
Epoch 3/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.2500 - loss: 2.8565
Epoch 4/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2500 - loss: 2.8377
Epoch 5/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2500 - loss: 2.8142
Epoch 6/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2500 - loss: 2.7831
Epoch 7/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2500 - loss: 2.7399
Epoch 8/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.2500 - loss: 2.6768
Epoch 9/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.2500 - loss: 2.5813
Epoch 10/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2500 - loss: 2.4395
Epoch 11/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.2500 - loss: 2.2848
Epoch 12/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2500 -

## Cell 6: Test Seq2Seq Prediction


In [ ]:
reverse_target_word_index = {index: word for word, index in target_tokenizer.word_index.items()}

sample = "i am happy"
sample_seq = input_tokenizer.texts_to_sequences([sample])
sample_seq = pad_sequences(sample_seq, maxlen=encoder_max_len, padding="post")

start_seq = target_tokenizer.texts_to_sequences(["<start>"])[0]
decoder_seq = pad_sequences([start_seq], maxlen=decoder_max_len, padding="post")

prediction = seq2seq_model.predict([sample_seq, decoder_seq])
predicted_ids = np.argmax(prediction[0], axis=-1)

predicted_words = [
    reverse_target_word_index.get(word_id, "")
    for word_id in predicted_ids
    if word_id != 0
]

print("Input sentence:", sample)
print("Predicted words:", " ".join(predicted_words))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Input sentence: i am happy
Predicted words: je suis heureux <end>


## Cell 7: Create Visual Attention Dataset


In [ ]:
num_samples = 1000
image_height = 8
image_width = 8
channels = 1

X = np.random.rand(num_samples, image_height, image_width, channels).astype("float32")

top_left_sum = X[:, :4, :4, :].sum(axis=(1, 2, 3))
bottom_right_sum = X[:, 4:, 4:, :].sum(axis=(1, 2, 3))
y = (bottom_right_sum > top_left_sum).astype("float32")

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1000, 8, 8, 1)
y shape: (1000,)


## Cell 8: Build Visual Attention Model


In [ ]:
image_inputs = Input(shape=(image_height, image_width, channels), name="image_inputs")
patches = Reshape((image_height * image_width, channels), name="flatten_pixels")(image_inputs)

attention_logits = Dense(1, name="attention_logits")(patches)
attention_scores = Softmax(axis=1, name="attention_scores")(attention_logits)
attended_pixels = patches * attention_scores
context = Lambda(lambda x: tf.reduce_sum(x, axis=1), name="attention_context")(attended_pixels)
outputs = Dense(1, activation="sigmoid", name="classifier")(context)

visual_attention_model = Model(image_inputs, outputs)
visual_attention_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
visual_attention_model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_inputs        │ (None, 8, 8, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_pixels      │ (None, 64, 1)     │          0 │ image_inputs[0][… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_logits    │ (None, 64, 1)     │          2 │ flatten_pixels[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_scores    │ (None, 64, 1)     │          0 │ attention_logits… │
│ (Softmax)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 64, 1)     │          0 │ flatten_pixels[0… │
│ (Multiply)          │                   │            │ attention_scores… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_context   │ (None, 1)         │          0 │ multiply_1[0][0]  │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier (Dense)  │ (None, 1)         │          2 │ attention_contex… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4 (16.00 B)

 Trainable params: 4 (16.00 B)

 Non-trainable params: 0 (0.00 B)

## Cell 9: Train Visual Attention Model


In [ ]:
visual_attention_model.fit(
    X,
    y,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
)


Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4900 - loss: 0.7545 - val_accuracy: 0.4200 - val_loss: 0.7983
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4900 - loss: 0.7487 - val_accuracy: 0.4200 - val_loss: 0.7904
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4900 - loss: 0.7433 - val_accuracy: 0.4200 - val_loss: 0.7829
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4900 - loss: 0.7384 - val_accuracy: 0.4200 - val_loss: 0.7760
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4900 - loss: 0.7339 - val_accuracy: 0.4200 - val_loss: 0.7695
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4900 - loss: 0.7298 - val_accuracy: 0.4200 - val_loss: 0.7635
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4900 - loss: 0.7260 - val_accuracy: 0.4200 - val_loss: 0.7579
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4900 - loss: 0.7226 - val_accuracy: 0.4200 - val_loss

## Cell 10: Test Visual Attention Model


In [ ]:
test_image = np.random.rand(1, image_height, image_width, channels).astype("float32")
prediction = visual_attention_model.predict(test_image)[0][0]

print(f"Prediction score: {prediction:.4f}")
print("Class 1 means bottom-right area has stronger signal than top-left area.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Prediction score: 0.6112
Class 1 means bottom-right area has stronger signal than top-left area.
